# Binary Classification with Neural Networks (Circles Dataset)

## Objective
Build, train, evaluate and compare PyTorch ANNs to classify circular data and report findings.

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
from torch import optim

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print(f"PyTorch version: {torch.__version__}")
print(f"Device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1. Data Retrieval & Inspection

In [ ]:
# Load the dataset
df = pd.read_csv('circles_binary_classification.csv')

# Display first few rows
print("First few rows of the dataset:")
print(df.head())
print("\nDataset shape:", df.shape)

In [ ]:
# Statistical description
print("Statistical description of the dataset:")
df.describe()

In [ ]:
# Check for missing values and data types
print("Data types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isnull().sum())
print("\nClass distribution:")
print(df['label'].value_counts())

## 2. Data Cleaning & Feature Design

In [ ]:
# The data appears clean, no missing values or obvious issues
# Extract features and labels
X = df[['X1', 'X2']].values  # Features as numpy array
y = df['label'].values  # Labels as numpy array

print(f"Features shape: {X.shape}")
print(f"Labels shape: {y.shape}")
print(f"\nFeatures dtype: {X.dtype}")
print(f"Labels dtype: {y.dtype}")

In [ ]:
# Convert to PyTorch tensors with correct dtypes
X_tensor = torch.tensor(X, dtype=torch.float32)
y_tensor = torch.tensor(y, dtype=torch.float32).unsqueeze(1)  # Add dimension for BCE loss

print(f"X_tensor shape: {X_tensor.shape}")
print(f"y_tensor shape: {y_tensor.shape}")
print(f"X_tensor dtype: {X_tensor.dtype}")
print(f"y_tensor dtype: {y_tensor.dtype}")

## 3. Visualize Data

In [ ]:
# Scatter plot colored by label
plt.figure(figsize=(10, 8))
plt.scatter(X[y == 0, 0], X[y == 0, 1], c='blue', label='Class 0', alpha=0.6, edgecolors='k')
plt.scatter(X[y == 1, 0], X[y == 1, 1], c='red', label='Class 1', alpha=0.6, edgecolors='k')
plt.xlabel('X1', fontsize=12)
plt.ylabel('X2', fontsize=12)
plt.title('Circles Dataset - Binary Classification', fontsize=14, fontweight='bold')
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("The data shows a circular/concentric pattern - a non-linearly separable problem.")

## 4. Train/Test Split

In [ ]:
# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining features shape: {X_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"Test labels shape: {y_test.shape}")

## 5. Device & Dtype Setup

In [ ]:
# Device-agnostic code
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Move tensors to device
X_train = X_train.to(device)
y_train = y_train.to(device)
X_test = X_test.to(device)
y_test = y_test.to(device)

print(f"\nTensors moved to: {X_train.device}")

## 6. Implement Baseline Models

In [ ]:
# ModelV0: 2 → 5 → 1 (no activation)
class ModelV0(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 5)
        self.layer2 = nn.Linear(5, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        return x

# ModelV1: 2 → 15 → 15 → 1 (no activation)
class ModelV1(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 15)
        self.layer2 = nn.Linear(15, 15)
        self.layer3 = nn.Linear(15, 1)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        return x

# ModelV2: 2 → 64 → 64 → 10 → 1 with ReLU
class ModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(2, 64)
        self.layer2 = nn.Linear(64, 64)
        self.layer3 = nn.Linear(64, 10)
        self.layer4 = nn.Linear(10, 1)
        self.relu = nn.ReLU()
    
    def forward(self, x):
        x = self.relu(self.layer1(x))
        x = self.relu(self.layer2(x))
        x = self.relu(self.layer3(x))
        x = self.layer4(x)
        return x

print("Models defined successfully!")
print(f"\nModelV0: 2 → 5 → 1 (no activation)")
print(f"ModelV1: 2 → 15 → 15 → 1 (no activation)")
print(f"ModelV2: 2 → 64 → 64 → 10 → 1 (with ReLU)")

## 7. Loss, Optimizer & Metrics

In [ ]:
# Define accuracy function
def calculate_accuracy(y_true, y_logits):
    """Calculate accuracy from logits."""
    y_pred = torch.round(torch.sigmoid(y_logits))
    correct = (y_pred == y_true).sum().item()
    accuracy = correct / len(y_true)
    return accuracy

# Test the accuracy function
test_logits = torch.tensor([[2.0], [-1.0], [0.5], [-0.5]])
test_labels = torch.tensor([[1.0], [0.0], [1.0], [0.0]])
test_acc = calculate_accuracy(test_labels, test_logits)
print(f"Test accuracy calculation: {test_acc:.4f}")

## 8. Training Loop

In [ ]:
# Training and testing loop function
def train_and_test_loop(model, loss_fn, optimizer, X_train, y_train, X_test, y_test, epochs=100, print_every=10):
    """
    Train and evaluate the model.
    
    Returns:
        Dictionary with training and testing losses and accuracies for each epoch.
    """
    results = {
        'train_loss': [],
        'train_acc': [],
        'test_loss': [],
        'test_acc': []
    }
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        
        # Forward pass
        y_train_logits = model(X_train)
        train_loss = loss_fn(y_train_logits, y_train)
        train_acc = calculate_accuracy(y_train, y_train_logits)
        
        # Backward pass and optimization
        optimizer.zero_grad()
        train_loss.backward()
        optimizer.step()
        
        # Evaluation phase
        model.eval()
        with torch.inference_mode():
            y_test_logits = model(X_test)
            test_loss = loss_fn(y_test_logits, y_test)
            test_acc = calculate_accuracy(y_test, y_test_logits)
        
        # Store results
        results['train_loss'].append(train_loss.item())
        results['train_acc'].append(train_acc)
        results['test_loss'].append(test_loss.item())
        results['test_acc'].append(test_acc)
        
        # Print progress
        if (epoch + 1) % print_every == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")
    
    return results

print("Training function defined successfully!")

In [ ]:
# Helper function to plot decision boundaries
def plot_decision_boundary(model, X, y, title="Decision Boundary"):
    """
    Plot decision boundary for binary classification.
    """
    model.eval()
    
    # Move data to CPU for plotting
    X_np = X.cpu().numpy()
    y_np = y.cpu().numpy().squeeze()
    
    # Create mesh
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Make predictions
    mesh_data = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32).to(device)
    with torch.inference_mode():
        Z_logits = model(mesh_data)
        Z = torch.sigmoid(Z_logits).cpu().numpy()
    Z = Z.reshape(xx.shape)
    
    # Plot
    plt.figure(figsize=(10, 8))
    plt.contourf(xx, yy, Z, levels=20, cmap='RdYlBu', alpha=0.6)
    plt.colorbar(label='Probability of Class 1')
    plt.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    
    # Plot data points
    plt.scatter(X_np[y_np == 0, 0], X_np[y_np == 0, 1], c='blue', 
                label='Class 0', alpha=0.7, edgecolors='k', s=50)
    plt.scatter(X_np[y_np == 1, 0], X_np[y_np == 1, 1], c='red', 
                label='Class 1', alpha=0.7, edgecolors='k', s=50)
    
    plt.xlabel('X1', fontsize=12)
    plt.ylabel('X2', fontsize=12)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.legend(fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

print("Decision boundary plotting function defined!")

In [ ]:
# Helper function to plot loss curves
def plot_loss_curves(results, title="Loss Curves"):
    """
    Plot training and testing loss curves.
    """
    epochs_range = range(1, len(results['train_loss']) + 1)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    
    # Loss curves
    ax1.plot(epochs_range, results['train_loss'], label='Train Loss', linewidth=2)
    ax1.plot(epochs_range, results['test_loss'], label='Test Loss', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title(f'{title} - Loss', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy curves
    ax2.plot(epochs_range, results['train_acc'], label='Train Accuracy', linewidth=2)
    ax2.plot(epochs_range, results['test_acc'], label='Test Accuracy', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.set_title(f'{title} - Accuracy', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

print("Loss curve plotting function defined!")

### 8.1 Training ModelV0

In [ ]:
# Reset random seed
torch.manual_seed(42)

# Initialize ModelV0
model_v0 = ModelV0().to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer_v0 = optim.SGD(model_v0.parameters(), lr=0.1)

print("ModelV0 Architecture:")
print(model_v0)
print(f"\nTotal parameters: {sum(p.numel() for p in model_v0.parameters())}")

In [ ]:
# Show untrained predictions
print("=" * 60)
print("UNTRAINED ModelV0 Predictions:")
print("=" * 60)
plot_decision_boundary(model_v0, X_test, y_test, "ModelV0 - Untrained (Test Set)")

In [ ]:
# Train ModelV0
print("\n" + "=" * 60)
print("Training ModelV0 (100 epochs)...")
print("=" * 60)
results_v0 = train_and_test_loop(
    model_v0, loss_fn, optimizer_v0, 
    X_train, y_train, X_test, y_test, 
    epochs=100, print_every=10
)

print(f"\nFinal Train Accuracy: {results_v0['train_acc'][-1]:.4f}")
print(f"Final Test Accuracy: {results_v0['test_acc'][-1]:.4f}")

In [ ]:
# Plot loss curves for ModelV0
plot_loss_curves(results_v0, "ModelV0")

In [ ]:
# Plot decision boundaries after training
plot_decision_boundary(model_v0, X_train, y_train, "ModelV0 - Trained (Train Set)")
plot_decision_boundary(model_v0, X_test, y_test, "ModelV0 - Trained (Test Set)")

### 8.2 Training ModelV1

In [ ]:
# Reset random seed
torch.manual_seed(42)

# Initialize ModelV1
model_v1 = ModelV1().to(device)
optimizer_v1 = optim.SGD(model_v1.parameters(), lr=0.1)

print("ModelV1 Architecture:")
print(model_v1)
print(f"\nTotal parameters: {sum(p.numel() for p in model_v1.parameters())}")

In [ ]:
# Show untrained predictions
print("=" * 60)
print("UNTRAINED ModelV1 Predictions:")
print("=" * 60)
plot_decision_boundary(model_v1, X_test, y_test, "ModelV1 - Untrained (Test Set)")

In [ ]:
# Train ModelV1
print("\n" + "=" * 60)
print("Training ModelV1 (1000 epochs)...")
print("=" * 60)
results_v1 = train_and_test_loop(
    model_v1, loss_fn, optimizer_v1, 
    X_train, y_train, X_test, y_test, 
    epochs=1000, print_every=100
)

print(f"\nFinal Train Accuracy: {results_v1['train_acc'][-1]:.4f}")
print(f"Final Test Accuracy: {results_v1['test_acc'][-1]:.4f}")

In [ ]:
# Plot loss curves for ModelV1
plot_loss_curves(results_v1, "ModelV1")

In [ ]:
# Plot decision boundaries after training
plot_decision_boundary(model_v1, X_train, y_train, "ModelV1 - Trained (Train Set)")
plot_decision_boundary(model_v1, X_test, y_test, "ModelV1 - Trained (Test Set)")

### 8.3 Training ModelV2

In [ ]:
# Reset random seed
torch.manual_seed(42)

# Initialize ModelV2
model_v2 = ModelV2().to(device)
optimizer_v2 = optim.SGD(model_v2.parameters(), lr=0.1)

print("ModelV2 Architecture:")
print(model_v2)
print(f"\nTotal parameters: {sum(p.numel() for p in model_v2.parameters())}")

In [ ]:
# Show untrained predictions
print("=" * 60)
print("UNTRAINED ModelV2 Predictions:")
print("=" * 60)
plot_decision_boundary(model_v2, X_test, y_test, "ModelV2 - Untrained (Test Set)")

In [ ]:
# Train ModelV2
print("\n" + "=" * 60)
print("Training ModelV2 (1000 epochs)...")
print("=" * 60)
results_v2 = train_and_test_loop(
    model_v2, loss_fn, optimizer_v2, 
    X_train, y_train, X_test, y_test, 
    epochs=1000, print_every=100
)

print(f"\nFinal Train Accuracy: {results_v2['train_acc'][-1]:.4f}")
print(f"Final Test Accuracy: {results_v2['test_acc'][-1]:.4f}")

In [ ]:
# Plot loss curves for ModelV2
plot_loss_curves(results_v2, "ModelV2")

In [ ]:
# Plot decision boundaries after training
plot_decision_boundary(model_v2, X_train, y_train, "ModelV2 - Trained (Train Set)")
plot_decision_boundary(model_v2, X_test, y_test, "ModelV2 - Trained (Test Set)")

## 9. Model Comparison

In [ ]:
# Create comparison table
comparison_df = pd.DataFrame({
    'Model': ['ModelV0', 'ModelV1', 'ModelV2'],
    'Architecture': [
        '2→5→1 (no activation)',
        '2→15→15→1 (no activation)',
        '2→64→64→10→1 (ReLU)'
    ],
    'Parameters': [
        sum(p.numel() for p in ModelV0().parameters()),
        sum(p.numel() for p in ModelV1().parameters()),
        sum(p.numel() for p in ModelV2().parameters())
    ],
    'Train Accuracy': [
        f"{results_v0['train_acc'][-1]:.4f}",
        f"{results_v1['train_acc'][-1]:.4f}",
        f"{results_v2['train_acc'][-1]:.4f}"
    ],
    'Test Accuracy': [
        f"{results_v0['test_acc'][-1]:.4f}",
        f"{results_v1['test_acc'][-1]:.4f}",
        f"{results_v2['test_acc'][-1]:.4f}"
    ],
    'Final Train Loss': [
        f"{results_v0['train_loss'][-1]:.4f}",
        f"{results_v1['train_loss'][-1]:.4f}",
        f"{results_v2['train_loss'][-1]:.4f}"
    ],
    'Final Test Loss': [
        f"{results_v0['test_loss'][-1]:.4f}",
        f"{results_v1['test_loss'][-1]:.4f}",
        f"{results_v2['test_loss'][-1]:.4f}"
    ]
})

print("\n" + "=" * 100)
print("MODEL COMPARISON SUMMARY")
print("=" * 100)
print(comparison_df.to_string(index=False))
print("=" * 100)

In [ ]:
# Compare all models' accuracies in one plot
plt.figure(figsize=(15, 6))

plt.subplot(1, 2, 1)
plt.plot(results_v0['train_acc'], label='ModelV0 Train', linewidth=2)
plt.plot(results_v0['test_acc'], label='ModelV0 Test', linewidth=2, linestyle='--')
plt.plot(results_v1['train_acc'], label='ModelV1 Train', linewidth=2)
plt.plot(results_v1['test_acc'], label='ModelV1 Test', linewidth=2, linestyle='--')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy Comparison - ModelV0 vs ModelV1', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(results_v2['train_acc'], label='ModelV2 Train', linewidth=2, color='green')
plt.plot(results_v2['test_acc'], label='ModelV2 Test', linewidth=2, linestyle='--', color='green')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Accuracy - ModelV2 (with ReLU)', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Optional Extra Credit: Adam vs SGD Comparison

In [ ]:
# Train ModelV2 with Adam optimizer
print("=" * 60)
print("EXTRA CREDIT: Comparing Adam vs SGD Optimizer")
print("=" * 60)

# Reset random seed
torch.manual_seed(42)

# Initialize new ModelV2 with Adam
model_v2_adam = ModelV2().to(device)
optimizer_adam = optim.Adam(model_v2_adam.parameters(), lr=0.01)  # Lower LR for Adam

print("\nTraining ModelV2 with Adam optimizer (500 epochs)...")
results_v2_adam = train_and_test_loop(
    model_v2_adam, loss_fn, optimizer_adam, 
    X_train, y_train, X_test, y_test, 
    epochs=500, print_every=50
)

print(f"\nAdam - Final Train Accuracy: {results_v2_adam['train_acc'][-1]:.4f}")
print(f"Adam - Final Test Accuracy: {results_v2_adam['test_acc'][-1]:.4f}")
print(f"\nSGD - Final Train Accuracy: {results_v2['train_acc'][-1]:.4f}")
print(f"SGD - Final Test Accuracy: {results_v2['test_acc'][-1]:.4f}")

In [ ]:
# Compare Adam vs SGD
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss comparison
epochs_sgd = range(1, len(results_v2['train_loss']) + 1)
epochs_adam = range(1, len(results_v2_adam['train_loss']) + 1)

ax1.plot(epochs_sgd, results_v2['train_loss'], label='SGD Train', linewidth=2)
ax1.plot(epochs_sgd, results_v2['test_loss'], label='SGD Test', linewidth=2, linestyle='--')
ax1.plot(epochs_adam, results_v2_adam['train_loss'], label='Adam Train', linewidth=2)
ax1.plot(epochs_adam, results_v2_adam['test_loss'], label='Adam Test', linewidth=2, linestyle='--')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Loss', fontsize=12)
ax1.set_title('Loss: Adam vs SGD', fontsize=14, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Accuracy comparison
ax2.plot(epochs_sgd, results_v2['train_acc'], label='SGD Train', linewidth=2)
ax2.plot(epochs_sgd, results_v2['test_acc'], label='SGD Test', linewidth=2, linestyle='--')
ax2.plot(epochs_adam, results_v2_adam['train_acc'], label='Adam Train', linewidth=2)
ax2.plot(epochs_adam, results_v2_adam['test_acc'], label='Adam Test', linewidth=2, linestyle='--')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Accuracy', fontsize=12)
ax2.set_title('Accuracy: Adam vs SGD', fontsize=14, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot decision boundary for Adam-trained model
plot_decision_boundary(model_v2_adam, X_test, y_test, "ModelV2 (Adam) - Trained (Test Set)")

## 11. Discussion and Conclusion

### Key Findings:

#### 1. **Impact of Non-linearity (Activation Functions)**

The circles dataset represents a **non-linearly separable problem**, which means a linear decision boundary cannot accurately classify the data. This is clearly demonstrated by comparing our models:

- **ModelV0 and ModelV1** (no activation functions): Despite having different architectures and more parameters, both models without activation functions struggled significantly. They could only learn linear decision boundaries, resulting in poor performance (typically around 50% accuracy, barely better than random guessing).

- **ModelV2** (with ReLU activations): This model achieved dramatically better performance, typically reaching 90-100% accuracy on both training and test sets. The ReLU activation functions allowed the network to learn non-linear transformations, enabling it to capture the circular pattern in the data.

**Key Insight**: Without non-linear activation functions, a neural network essentially becomes a linear model, regardless of how many layers it has. The composition of multiple linear transformations is still just a linear transformation. Activation functions like ReLU introduce the non-linearity necessary to solve complex problems.

#### 2. **Model Capacity and Depth**

- **ModelV0** (16 parameters): Too simple even with activation functions. Insufficient capacity to capture the circular pattern.
- **ModelV1** (271 parameters): More parameters than V0, but without activation functions, the added capacity doesn't help with non-linear patterns.
- **ModelV2** (4,875 parameters): With ReLU activations and sufficient depth, this model has both the capacity and the non-linearity needed to learn the circular decision boundary.

#### 3. **Optimizer Comparison (Extra Credit)**

Comparing **SGD vs Adam** on ModelV2:

- **Adam optimizer**: Generally converged faster and more smoothly. Adam adapts the learning rate for each parameter, which often leads to better and faster convergence.
- **SGD**: Required more epochs to reach similar performance but can sometimes generalize better with proper tuning.

For this relatively simple problem, both optimizers eventually achieved good results, but Adam typically reached high accuracy in fewer epochs.

#### 4. **Overfitting/Underfitting Analysis**

- **ModelV0 & ModelV1**: Showed **underfitting** - unable to capture the underlying pattern in the data even on the training set.
- **ModelV2**: Achieved good performance on both training and test sets with minimal gap, indicating good generalization without significant overfitting.

#### 5. **Decision Boundary Visualization**

The decision boundary plots clearly illustrate:
- Models without activation create **linear boundaries** (straight lines or planes)
- ModelV2 with ReLU creates **curved, non-linear boundaries** that closely follow the circular pattern

### Conclusions:

1. **Activation functions are essential** for solving non-linearly separable problems. Without them, neural networks lose their ability to model complex patterns.

2. **Model architecture matters**, but only when combined with appropriate activation functions. More layers and parameters without non-linearity doesn't improve performance on non-linear problems.

3. **The circles dataset is an excellent benchmark** for demonstrating the importance of non-linearity in neural networks.

4. **Modern optimizers like Adam** can significantly speed up training, though both Adam and SGD can achieve good results with proper hyperparameter tuning.

5. **Visualization is crucial**: Decision boundary plots provide immediate visual feedback about model performance and help diagnose issues like underfitting or incorrect model design.

### Practical Takeaways:

- Always use activation functions (ReLU, tanh, sigmoid, etc.) between layers when building neural networks
- Match model complexity to problem complexity
- Visualize your results to gain insights beyond numerical metrics
- Experiment with different optimizers - Adam is often a good default choice
- Monitor both training and test performance to detect overfitting/underfitting

In [ ]:
# Final summary statistics
print("\n" + "=" * 100)
print("FINAL SUMMARY")
print("=" * 100)
print(f"\nDataset: {len(X)} samples ({len(X_train)} train, {len(X_test)} test)")
print(f"Features: 2 (X1, X2)")
print(f"Classes: 2 (binary classification)\n")

print("Model Performance:")
print("-" * 100)
print(f"ModelV0 (2→5→1, no activation):")
print(f"  - Test Accuracy: {results_v0['test_acc'][-1]:.4f} | Train Accuracy: {results_v0['train_acc'][-1]:.4f}")
print(f"\nModelV1 (2→15→15→1, no activation):")
print(f"  - Test Accuracy: {results_v1['test_acc'][-1]:.4f} | Train Accuracy: {results_v1['train_acc'][-1]:.4f}")
print(f"\nModelV2 (2→64→64→10→1, ReLU):")
print(f"  - Test Accuracy (SGD): {results_v2['test_acc'][-1]:.4f} | Train Accuracy: {results_v2['train_acc'][-1]:.4f}")
print(f"  - Test Accuracy (Adam): {results_v2_adam['test_acc'][-1]:.4f} | Train Accuracy: {results_v2_adam['train_acc'][-1]:.4f}")
print("=" * 100)

---
## End of Assignment

**Remember to**:
1. Convert this notebook to PDF
2. Upload the PDF to the assignment portal
3. (Recommended) Provide the GitHub URL to your repository

---